# Phase M2-A2 — SwinUNETR Fine-Tuning & ViT Embedding (MU-Glioma-Post)## 3 output channels: WT / TC / ET (Same label convention as BraTS 2024)**Starting weights**: BraTS 2021 Fold 1 pretrained (original — NOT BraTS 2024 fine-tuned)**Data**: 596 MU-Glioma-Post scans (203 patients, up to 6 timepoints each)**Architecture**: SwinUNETR-Base, ~62M params, feature_size=48### Two-Stage Training Strategy:1. **Stage 1**: Dice-only fine-tuning (28 epochs) — adapt features to MU-Glioma anatomy2. **Stage 2**: Triplet contrastive training (10 epochs) — make embeddings temporally aware### Kaggle Datasets Required:1. `mu-glioma-post` — MU-Glioma NIfTI imaging data2. `swinunetr-brats2021-fold1` — SwinUNETR pretrained weights (`swinunetr_fold1.pt`)3. `mu-glioma-m1-outputs` — M1 pipeline outputs4. *(Optional)* `mu-glioma-m2-nnunet` — Previous M2_A1 output (for checkpoint recovery)### Pipeline:1. Load BraTS 2021 pretrained SwinUNETR2. Stage 1: Fine-tune segmentation (Dice loss, partial encoder freeze)3. Stage 2: Temporal contrastive training (triplet loss on encoder)4. Extract embeddings: octant(8C) + region(3C) + vol(9) per scan (C=384 from layers3)5. Save embeddings + spatial tokens for downstream TaViT V3

In [ ]:
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'swinunetr'
PATCH       = (128, 128, 128)
REGIONS     = ['WT', 'TC', 'ET']
OUTPUT_ROOT = Path('/kaggle/working/phase_m2_swinunetr')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Model: {MODEL_NAME} | Patch: {PATCH} | Regions: {REGIONS}')

In [ ]:
import subprocess, sys, json, time, math, shutil, gc, random
import numpy as np
import torch
import torch.nn.functional as F

try:
    import monai
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai[all]', '-q'])
    import monai

import monai.transforms as T
from monai.data import Dataset, CacheDataset, DataLoader
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.transforms import MapTransform
from monai.networks.nets import SwinUNETR

set_determinism(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MONAI {monai.__version__} | PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | Total memory: {total_mem:.1f} GB')

usage = shutil.disk_usage('/kaggle/working')
free_gb = usage.free / 1e9
print(f'Disk free: {free_gb:.1f} GB (need ~3GB for checkpoints)')

In [ ]:
# MU-Glioma-Post labels (IDENTICAL to BraTS 2024):
#   0 = Background, 1 = NETC, 2 = SNFH, 3 = ET, 4 = RC
# WT=1+2+3, TC=1+3, ET=3, RC=4(excluded)

class ConvertToMultiChannelBrats3Chd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT
                (img==1)|(img==3),           # TC
                img==3,                      # ET
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

print('MU-Glioma Label mapping: WT=1+2+3 | TC=1+3 | ET=3')

In [ ]:
# MU-Glioma Data Discovery
# Kaggle renames .nii.gz to .nii_gz -- symlink trick restores proper extension
import nibabel as nib
import os

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        rel = nii_gz.relative_to(data_dir)
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / rel.parent / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    nii_gz_files = list(ds_dir.rglob('*.nii_gz'))
    if nii_gz_files:
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks from {ds_dir.name}')

scan_index_path = None
splits_path = None
master_csv_path = None
for p in DATA_ROOT.rglob('scan_index.json'):
    scan_index_path = p; break
for p in DATA_ROOT.rglob('data_splits.json'):
    splits_path = p; break
for p in DATA_ROOT.rglob('mu_glioma_master.csv'):
    master_csv_path = p; break

print(f'scan_index.json: {scan_index_path}')
print(f'data_splits.json: {splits_path}')
print(f'master_csv.csv: {master_csv_path}')

if scan_index_path is None:
    raise RuntimeError('scan_index.json not found')

with open(scan_index_path) as f:
    scan_index = json.load(f)
scans_dict = scan_index['scans']
print(f'Total scans in index: {len(scans_dict)}')

if splits_path:
    with open(splits_path) as f:
        splits = json.load(f)
    train_pids = set(splits['train'])
    val_pids   = set(splits['val'])
    test_pids  = set(splits['test'])
    print(f'Splits: Train={len(train_pids)} | Val={len(val_pids)} | Test={len(test_pids)}')
else:
    all_pids = sorted(set(s['patient_id'] for s in scans_dict.values()))
    n80 = int(0.8 * len(all_pids))
    train_pids = set(all_pids[:n80])
    val_pids = set(all_pids[n80:])
    test_pids = set()
    print(f'No splits file, 80/20: Train {len(train_pids)} | Val {len(val_pids)}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for p in search_root.rglob('PatientID_0003'):
        if p.is_dir():
            NIFTI_ROOT = p.parent
            break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    for search_root in [SYMLINK_DIR, DATA_ROOT]:
        if not search_root.exists(): continue
        for pat in ['*brain_t1c.nii.gz', '*brain_t1c.nii_gz', '*brain_t1c.nii']:
            for p in search_root.rglob(pat):
                NIFTI_ROOT = p.parent.parent.parent
                break
            if NIFTI_ROOT: break
        if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('Could not find MU-Glioma NIfTI data')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

all_scans = []
for scan_id, scan_data in scans_dict.items():
    pid = scan_data['patient_id']
    tp  = scan_data['timepoint']
    tp_dir = NIFTI_ROOT / pid / f'Timepoint_{tp}'
    paths = {}
    for mod in ['t1c', 't1n', 't2f', 't2w']:
        candidates = list(tp_dir.glob(f'*brain_{mod}*')) if tp_dir.exists() else []
        if candidates:
            paths[mod] = str(candidates[0])
    mask_candidates = list(tp_dir.glob('*tumorMask*')) if tp_dir.exists() else []
    if mask_candidates:
        paths['seg'] = str(mask_candidates[0])
    if not all(m in paths for m in ['t1c', 't1n', 't2f', 't2w', 'seg']):
        continue
    entry = {
        'scan_id': scan_id, 'patient_id': pid, 'timepoint': str(tp),
        't1n': paths['t1n'], 't1c': paths['t1c'],
        't2w': paths['t2w'], 't2f': paths['t2f'],
        'seg': paths['seg'],
        'split': 'train' if pid in train_pids else ('val' if pid in val_pids else 'test'),
    }
    all_scans.append(entry)

train_scans = [s for s in all_scans if s['split'] == 'train']
val_scans   = [s for s in all_scans if s['split'] == 'val']
print(f'Resolved: {len(all_scans)} total scans')
print(f'Train: {len(train_scans)} | Val: {len(val_scans)}')

if all_scans:
    test_path = all_scans[0]['t1c']
    print(f'Spot-check: {test_path}')
    print(f'  exists={Path(test_path).exists()}, size={Path(test_path).stat().st_size}')
    try:
        img = nib.load(test_path)
        print(f'  nibabel OK: shape={img.shape}')
    except Exception as e:
        print(f'  nibabel FAIL: {e}')

In [ ]:
patch = list(PATCH)
train_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.RandFlipd(keys=['image','label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=0.3),
    T.SpatialPadd(keys=['image','label'], spatial_size=patch),
    T.RandCropByPosNegLabeld(keys=['image','label'], label_key='label',
        spatial_size=patch, pos=2, neg=1, num_samples=1, image_key='image', image_threshold=0),  # 1 patch/scan → 1 per GPU with DataParallel
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('Transforms ready (3-channel: WT/TC/ET)')

In [ ]:
# Validation: existence + size check (symlinks fix .nii_gz extension)
def validate_scan(s):
    try:
        for key in ['t1n','t1c','t2w','t2f','seg']:
            p = Path(s[key])
            if not p.exists(): return False
            if p.stat().st_size < 1024: return False
        return True
    except Exception:
        return False

def build_dicts(scan_list, label=''):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  {label}Skipped {len(bad)} missing: {bad[:3]}{"..." if len(bad)>3 else ""}')
    return dicts

print('Validating scans...')
train_dicts = build_dicts(train_scans, 'Train: ')
val_dicts   = build_dicts(val_scans, 'Val: ')
all_dicts   = build_dicts(all_scans, 'All: ')
print(f'Train dicts: {len(train_dicts)} | Val dicts: {len(val_dicts)} | All dicts: {len(all_dicts)}')
if train_dicts:
    print(f'  Sample: {train_dicts[0]["image"][0]}')
    print(f'  Exists: {Path(train_dicts[0]["image"][0]).exists()}')

In [ ]:
# ═══════════ SwinUNETR Model + Pretrained Weights ═══════════
print('='*55)
print('  Loading SwinUNETR (BraTS 2021 Fold 1 — original)')
print('='*55)

model = SwinUNETR(
    in_channels=4,
    out_channels=3,
    feature_size=48,
    use_checkpoint=True,
)
print(f'  SwinUNETR created: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

# Find pretrained weights on Kaggle
weight_path = None
for p in Path('/kaggle/input').rglob('swinunetr_fold1.pt'):
    weight_path = p; break
if weight_path is None:
    for p in Path('/kaggle/input').rglob('model.pt'):
        if 'fold1' in str(p).lower() or 'swin' in str(p).lower():
            weight_path = p; break

if weight_path:
    print(f'  Found weights: {weight_path}')
    try:
        ckpt = torch.load(weight_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(weight_path, map_location='cpu')
    state = ckpt.get('state_dict', ckpt)
    own   = model.state_dict()
    compat = {k: v for k, v in state.items()
              if k in own and own[k].shape == v.shape}
    model.load_state_dict({**own, **compat}, strict=False)
    print(f'  Pretrained: {len(compat)}/{len(own)} layers loaded')
    if len(compat) < len(own) // 2:
        print('  WARNING: <50% layers matched — check weight file')
else:
    print('  WARNING: No pretrained weights found — training from scratch')

# ── Multi-GPU: DataParallel across both T4s (2×15 GB = 30 GB VRAM) ──
# No freeze needed — each GPU handles 1 sample, activations split across GPUs
# → Each GPU sees ~8 GB peak usage for full 62.2M backward pass
for param in model.parameters():
    param.requires_grad = True
frozen = 0  # nothing frozen — full fine-tune on both GPUs

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = torch.nn.DataParallel(model)
    print(f'  ✅ DataParallel: {n_gpus} GPUs ({n_gpus*15}GB VRAM total)')
else:
    print(f'  ⚠ Single GPU detected — consider enabling T4×2 session')

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total: {n_total/1e6:.1f}M | Trainable: {n_train/1e6:.1f}M | Frozen params: {frozen}')
model = model.to(device)
print(f'  Model on {device}')

In [ ]:
from torch.cuda.amp import GradScaler, autocast

CKPT_DIR    = OUTPUT_ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH   = CKPT_DIR / 'swinunetr_best.pth'
LATEST_PATH = CKPT_DIR / 'swinunetr_latest.pth'

# ── WARM RESTART FLAG ──────────────────────────────────────
# True  → load best weights + restart OneCycleLR from step 0
# False → normal resume / skip if already complete
WARM_RESTART = True

# Step 1: Recover checkpoints from Kaggle input (both or either)
for src_f in sorted(Path('/kaggle/input').rglob('swinunetr_best.pth')):
    if not BEST_PATH.exists():
        shutil.copy2(src_f, BEST_PATH)
        print(f'  Recovered BEST from {src_f}')
    break
for src_f in sorted(Path('/kaggle/input').rglob('swinunetr_latest.pth')):
    if not LATEST_PATH.exists():
        shutil.copy2(src_f, LATEST_PATH)
        print(f'  Recovered LATEST from {src_f}')
    break

# Step 2: Apply WARM_RESTART AFTER recovery — overwrite epoch to -1
# This fires whether LATEST came from input or was just created
if WARM_RESTART and BEST_PATH.exists():
    ckpt = torch.load(BEST_PATH, map_location='cpu')
    prev_dice = ckpt.get('best_dice', 0)
    ckpt['epoch'] = -1          # epoch=-1 → train_model restarts from ep 0
    torch.save(ckpt, LATEST_PATH)
    print(f'  🔁 WARM RESTART: best weights loaded (Dice={prev_dice:.4f}) → schedule restarted')
elif LATEST_PATH.exists():
    ckpt = torch.load(LATEST_PATH, map_location='cpu')
    ep   = ckpt.get('epoch', 0)
    print(f'  ✅ Resume from epoch {ep} (WARM_RESTART=False)')
elif not BEST_PATH.exists() and not LATEST_PATH.exists():
    print('  ⚠ No checkpoints found — training from BraTS2021 pretrained weights')

# OneCycleLR replaces manual get_lr — peaks at max_lr in first 10% of steps

def safe_loader_iter(loader):
    it = iter(loader)
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream','worker','corrupt','truncat',
            'ImageFileError','not a gzip','applying transform']
    def _err_chain(e):
        """Flatten exception chain to one string for keyword matching."""
        parts = []
        while e is not None:
            parts.append(str(e))
            e = getattr(e, '__cause__', None) or getattr(e, '__context__', None)
        return ' | '.join(parts)
    while True:
        try: yield next(it)
        except StopIteration: return
        except Exception as e:
            if any(k in _err_chain(e) for k in SKIP): continue
            raise

def train_model(model, lr=1e-4, epochs=35, patience=15, val_interval=2):
    start_ep, best_dice, mlog = 0, 0.0, {'dice':[],'per_region':[],'loss':[]}
    if LATEST_PATH.exists():
        lc = torch.load(LATEST_PATH, map_location='cpu')
        _mbase = model.module if isinstance(model, torch.nn.DataParallel) else model
        _mbase.load_state_dict(lc['model'])
        saved_ep = lc.get('epoch', 0)
        if saved_ep == -1:
            # Warm restart: weights loaded, but schedule starts fresh from ep 0
            start_ep  = 0
            best_dice = lc.get('best_dice', 0.7923)
            print(f'🔁 WARM RESTART: weights loaded (prev best={best_dice:.4f}) → training from ep 0')
        else:
            start_ep  = saved_ep + 1
            best_dice = lc.get('best_dice', 0)
            mlog      = lc.get('metrics', mlog)
            print(f'Resumed from epoch {start_ep-1}, best_dice={best_dice:.4f}')
        if start_ep >= epochs:
            return model, best_dice, mlog

    # DiceCELoss: CE term never saturates at high Dice → stronger gradient signal
    # Critical when fine-tuning from Dice=0.79 where pure DiceLoss gradient is tiny
    # EXACT nnUNet loss: DiceLoss only, sigmoid activation, no smoothing on numerator
    # Matches M2_A1 nnUNet Finetune exactly → same loss landscape, same convergence signal
    loss_fn     = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    _m = model.module if isinstance(model, torch.nn.DataParallel) else model
    optimizer   = torch.optim.AdamW(filter(lambda p: p.requires_grad, _m.parameters()),
                                     lr=lr, weight_decay=1e-4)
    # PolyLR: starts at full lr immediately (NO warm-up) → safe for fine-tuning
    # Matches nnUNet's implicit schedule (constant LR = PolyLR with power=0)
    # Power=0.9 gives steady linear-ish decay; no warm-up means pretrained weights
    # are never perturbed by a sudden LR spike
    steps_per_epoch = 215  # 429 scans / batch_size=2 = 215 (num_samples=1 → 1 patch/GPU)
    total_steps = epochs * steps_per_epoch
    scheduler   = torch.optim.lr_scheduler.PolynomialLR(
                    optimizer,
                    total_iters = total_steps,
                    power       = 0.9)
    # Fast-forward scheduler if resuming
    if start_ep > 0:
        for _ in range(start_ep * steps_per_epoch):
            scheduler.step()
        print(f'  Scheduler fast-forwarded to epoch {start_ep}')
    scaler      = GradScaler()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    no_improve  = 0
    t0          = time.time()

    try:
        # RandCropByPosNegLabeld(num_samples=2) stores 2 patches per scan → doubles RAM
        # Empirical: crashed at 328 scans × 79 MB/scan ≈ 26 GB on 30 GB machine
        # Safe: 40% × 429 = 172 scans × 79 MB ≈ 13.6 GB → leaves plenty of headroom
        train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=0.4, num_workers=0)
        n_cached = int(len(train_dicts) * 0.4)
        print(f'CacheDataset: 40% ({n_cached} scans ~{n_cached*79//1024} GB) — safe for 30 GB RAM')
    except Exception as _ce:
        print(f'Cache failed ({_ce}), using Dataset')
        train_ds = Dataset(train_dicts, train_transforms)
    val_ds = Dataset(val_dicts, val_transforms)

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

    for ep in range(start_ep, epochs):
        model.train()

        ep_loss, n_ok, n_bad = 0.0, 0, 0
        _err_sample = []  # capture first 3 errors for diagnosis
        for batch in safe_loader_iter(train_loader):
            try:
                imgs = batch['image'].to(device)
                lbls = batch['label'].to(device)
                optimizer.zero_grad()
                with autocast():
                    loss = loss_fn(model(imgs), lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                ep_loss += loss.item(); n_ok += 1
            except Exception as _e:
                n_bad += 1
                if len(_err_sample) < 3:
                    _err_sample.append(f'{type(_e).__name__}: {str(_e)[:120]}')
            finally:
                scheduler.step()  # always step — outside try so LR advances regardless
        avg_loss = ep_loss / max(n_ok, 1)
        if _err_sample and ep == start_ep:  # print once on first epoch
            print(f'  ⚠ Sample errors (first {len(_err_sample)}): {_err_sample}')
        bad_str  = f' | skipped {n_bad}' if n_bad else ''  
        mlog['loss'].append(avg_loss)

        if (ep+1) % val_interval == 0 or ep == epochs-1:
            model.eval(); dice_metric.reset()
            with torch.no_grad():
                for vb in safe_loader_iter(val_loader):
                    try:
                        vo = sliding_window_inference(vb['image'].to(device), list(PATCH), 2, model, overlap=0.25)
                        dice_metric((torch.sigmoid(vo)>0.5).float(), vb['label'].to(device))
                    except Exception: pass
            dv = dice_metric.aggregate(); md = dv.mean().item()
            pr = [round(dv[i].item(),4) for i in range(3)]
            mlog['dice'].append(md); mlog['per_region'].append(pr)
            tag = ' NEW BEST' if md > best_dice else ''
            print(f'Ep {ep:3d} | lr={scheduler.get_last_lr()[0]:.2e} | L={avg_loss:.4f} | Dice={md:.4f} WT={pr[0]:.3f} TC={pr[1]:.3f} ET={pr[2]:.3f} | {(time.time()-t0)/60:.1f}m{tag}{bad_str}')
            if md > best_dice:
                best_dice = md; no_improve = 0
                torch.save({'model': (model.module if isinstance(model, torch.nn.DataParallel) else model).state_dict(), 'epoch': ep, 'best_dice': best_dice}, BEST_PATH)
            else:
                no_improve += val_interval
            _ms_l = model.module if isinstance(model, torch.nn.DataParallel) else model
            torch.save({'model': _ms_l.state_dict(), 'epoch': ep,
                        'best_dice': best_dice, 'metrics': mlog}, LATEST_PATH)
            if no_improve >= patience:
                print(f'Early stopping at epoch {ep}')
                break
        else:
            print(f'Ep {ep:3d} | lr={scheduler.get_last_lr()[0]:.2e} | L={avg_loss:.4f} | {(time.time()-t0)/60:.1f}m{bad_str}')

    if BEST_PATH.exists():
        _mbase = model.module if isinstance(model, torch.nn.DataParallel) else model
        _mbase.load_state_dict(torch.load(BEST_PATH, map_location='cpu')['model'])
        print(f'Loaded BEST checkpoint → Dice {best_dice:.4f}')

    for f in [BEST_PATH, LATEST_PATH]:
        dest = OUTPUT_ROOT / f.name
        if not dest.exists(): shutil.copy2(f, dest)

    return model, best_dice, mlog

model, best_dice, metrics_log = train_model(model)
print(f'\nStage 1 complete. Best Dice: {best_dice:.4f}')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(exist_ok=True)
REGION_COLORS = {'WT': ('red','lightcoral'), 'TC': ('green','lightgreen'), 'ET': ('blue','lightskyblue')}

def visualize_3d_predictions(model, n_samples=5):
    model.eval()
    vis_dicts = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    print(f'  Visualising {len(vis_dicts)} random val patients')
    dice_metric = DiceMetric(include_background=True, reduction='none')
    vis_ds = Dataset(vis_dicts, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    for idx, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()
            vi = batch['image'].to(device)
            vl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(vi, list(PATCH), 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float()
            dice_metric.reset(); dice_metric(pred_bin, vl)
            dv = dice_metric.aggregate()[0].cpu().numpy()
            pid = batch.get('patient_id', ['?'])[0]
            def fmt(v): return 'NaN' if np.isnan(v) else f'{v:.3f}'
            dstr = '  '.join(f'{r}={fmt(dv[j])}' for j,r in enumerate(REGIONS))
            title = f'{pid} | {dstr}  Mean={float(np.nanmean(dv)):.3f}'
            pred_np = pred_bin.squeeze(0).cpu().numpy()
            gt_np = vl.squeeze(0).cpu().numpy()
            del vi, vl, vo, pred_bin; gc.collect(); torch.cuda.empty_cache()
            STEP = 3
            fig = plt.figure(figsize=(18, 7))
            fig.suptitle(title, fontsize=10, y=1.01)
            for col, (arr, col_title) in enumerate([(gt_np, 'Ground Truth'), (pred_np, 'SwinUNETR Pred')]):
                ax = fig.add_subplot(1, 3, col+1, projection='3d')
                ax.set_title(col_title, fontsize=9)
                for ch, (region, (clr, _)) in enumerate(REGION_COLORS.items()):
                    vox = arr[ch]
                    coords = np.argwhere(vox[::STEP, ::STEP, ::STEP] > 0.5)
                    if len(coords) > 0:
                        ax.scatter(coords[:,2]*STEP, coords[:,1]*STEP, coords[:,0]*STEP,
                                   c=clr, alpha=0.3, s=2, label=region)
                ax.legend(fontsize=7)
            ax3 = fig.add_subplot(1, 3, 3)
            colors_bar = [REGION_COLORS[r][0] for r in REGIONS]
            valid_dv = [dv[j] if not np.isnan(dv[j]) else 0 for j in range(3)]
            ax3.barh(REGIONS, valid_dv, color=colors_bar, edgecolor='black')
            ax3.set_xlim(0, 1); ax3.set_xlabel('Dice'); ax3.set_title('Per-Region Dice')
            for j, v in enumerate(valid_dv): ax3.text(v+0.01, j, f'{v:.3f}', va='center', fontsize=9)
            plt.tight_layout()
            plt.savefig(fig_dir / f'val_3d_{idx}_{pid}.png', dpi=150, bbox_inches='tight')
            plt.close()
            print(f'    {pid}: {dstr}')
        except Exception as e:
            print(f'    Error {idx}: {e}')
            gc.collect(); torch.cuda.empty_cache()

visualize_3d_predictions(model, n_samples=5)

In [ ]:
# ═══════════ ViT (SwinUNETR) Embedding Extraction ═══════════
# IDENTICAL strategy to BraTS Phase 3:
#   - Hook swinViT.layers3[0]: 8×8×8, C=384
#   - Octant spatial pooling (8C) + Mask-weighted region (3C) + Vol morph (9)
#   - Total: 8×384 + 3×384 + 9 = 4233-D per scan

def _is_corrupt(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files')
            return
        except Exception as e:
            if _is_corrupt(e): n_skip += 1; continue
            raise

def get_wt_bbox(lbl_feat, min_size=2):
    wt = lbl_feat[0]
    mask = (wt > 0.01).nonzero(as_tuple=False)
    if len(mask) < 1: return None
    z_min, y_min, x_min = mask.min(dim=0).values.tolist()
    z_max, y_max, x_max = mask.max(dim=0).values.tolist()
    h, w, d = wt.shape
    z0 = max(z_min-1, 0); z1 = min(z_max+2, h)
    y0 = max(y_min-1, 0); y1 = min(y_max+2, w)
    x0 = max(x_min-1, 0); x1 = min(x_max+2, d)
    if z1-z0 < min_size: z1 = min(z0+min_size, h)
    if y1-y0 < min_size: y1 = min(y0+min_size, w)
    if x1-x0 < min_size: x1 = min(x0+min_size, d)
    return (z0, z1, y0, y1, x0, x1)

def extract_embeddings(model):
    model.eval()
    _feats = {}; hooks = []

    sv = model.module.swinViT if isinstance(model, torch.nn.DataParallel) else model.swinViT
    target_layer = 'layers3'
    C_feat = 384
    if not hasattr(sv, target_layer):
        target_layer = 'layers2'; C_feat = 192
    layer_list = getattr(sv, target_layer)
    target = layer_list[0] if hasattr(layer_list,'__getitem__') and len(layer_list)>0 else layer_list
    def _hook(m, inp, out):
        feat = out[-1] if isinstance(out,(list,tuple)) else out
        _feats['feat'] = feat.detach()
    hooks.append(target.register_forward_hook(_hook))
    print(f'  Hook: swinViT.{target_layer}[0] → C_feat={C_feat}')

    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(exist_ok=True)
    embs, ids, tps = [], [], []
    spatial_tokens_list = []
    bboxes_list = []
    ds = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total = len(all_dicts)
    n_skip = 0; n_empty = 0
    t_start = time.time()

    OCT_DIM = 8 * C_feat; REG_DIM = 3 * C_feat; VOL_DIM = 9
    TOTAL_DIM = OCT_DIM + REG_DIM + VOL_DIM
    print(f'  Extraction: {total} scans → ~{TOTAL_DIM}-D')
    print(f'  ⚠️  Using MODEL PREDICTIONS (not GT) for volumes & ROI')
    print(f'  (octant={OCT_DIM} + region={REG_DIM} + vol={VOL_DIM})')

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = batch['timepoint'][0]
            try:
                img = batch['image'].to(device)
                lbl = batch['label'].to(device)
                img_p = F.interpolate(img, list(PATCH), mode='trilinear', align_corners=False)
                # GT label loaded but not used for embedding (using predicted mask instead)
                _feats.clear()
                # Get model prediction as mask (instead of GT)
                logits = model(img_p)
                pred_mask = (torch.sigmoid(logits) > 0.5).float()
                # Use predicted mask for volumes and ROI
                lbl_pred = pred_mask
                if 'feat' not in _feats:
                    n_skip += 1; continue
                feat = _feats['feat']
                C = feat.shape[1]
                h, w, d = feat.shape[2], feat.shape[3], feat.shape[4]
                if idx == 0:
                    C_feat = C; OCT_DIM = 8*C; REG_DIM = 3*C; TOTAL_DIM = OCT_DIM+REG_DIM+VOL_DIM
                    print(f'  Feature map: {tuple(feat.shape)} → C={C}')
                    print(f'  Embedding: {TOTAL_DIM}-D')

                lbl_feat = F.adaptive_avg_pool3d(lbl_pred, feat.shape[2:])
                wt_vol = float(lbl_pred[0, 0].sum().item())
                tc_vol = float(lbl_pred[0, 1].sum().item())
                et_vol = float(lbl_pred[0, 2].sum().item())

                feat_crop = None
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)
                if bbox is not None:
                    z0, z1, y0, y1, x0, x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]
                    oct_pooled = F.adaptive_avg_pool3d(feat_crop, (2, 2, 2))
                    oct_vec = oct_pooled[0].reshape(C, 8).T.reshape(-1)
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                    feat_flat = feat_crop[0].reshape(C, -1)
                    region_vecs = []
                    for ch in range(3):
                        mask = lbl_crop[0, ch].reshape(-1)
                        if float(mask.sum()) > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec)
                    roi_size = f'{z1-z0}×{y1-y0}×{x1-x0}'
                    sp_tok = feat_crop[0].reshape(C, -1).T.cpu().numpy()
                    bbox_entry = list(bbox)
                else:
                    oct_vec = torch.zeros(8 * C, device=device)
                    region_vecs = [torch.zeros(C, device=device) for _ in range(3)]
                    roi_size = 'empty'; n_empty += 1
                    sp_tok = np.zeros((1, C), dtype=np.float32)
                    bbox_entry = [0,0,0,0,0,0]

                log_wt = np.log1p(wt_vol); log_tc = np.log1p(tc_vol); log_et = np.log1p(et_vol)
                has_wt = 1.0 if wt_vol > 10 else 0.0
                has_tc = 1.0 if tc_vol > 10 else 0.0
                has_et = 1.0 if et_vol > 10 else 0.0
                vol_feat = torch.tensor([log_wt, log_tc, log_et, has_wt, has_tc, has_et,
                    min(tc_vol/(wt_vol+1e-6),1.0), min(et_vol/(wt_vol+1e-6),1.0), min(et_vol/(tc_vol+1e-6),1.0)], dtype=torch.float32)

                emb = torch.cat([oct_vec.cpu()] + [v.cpu() for v in region_vecs] + [vol_feat]).numpy()
                del img, lbl, img_p, feat, lbl_pred
                if feat_crop is not None: del feat_crop

                spatial_tokens_list.append(sp_tok)
                bboxes_list.append(bbox_entry)
                embs.append(emb); ids.append(pid); tps.append(tp)

                if (idx+1) % 50 == 0 or idx == 0:
                    elapsed = time.time() - t_start
                    rate = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total-idx-1) / max(rate, 1e-6)
                    print(f'  [{idx+1:4d}/{total}] {pid:<30} tp={tp} WT={wt_vol:6.0f}v ROI={roi_size} | {rate:.1f}/s ETA {remaining/60:.1f}m')
                elif (idx+1) % 10 == 0:
                    elapsed = time.time() - t_start
                    rate = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total-idx-1) / max(rate,1e-6)
                    print(f'  [{idx+1:4d}/{total}] done={len(embs)} | {rate:.1f}/s ETA {remaining/60:.1f}m')

            except Exception as e:
                print(f'  [{idx+1:4d}/{total}] ERROR {pid}: {str(e)[:70]}')
                n_skip += 1; continue

    for hk in hooks:
        try: hk.remove()
        except: pass

    print(f'  Done: {len(embs)}/{total} in {(time.time()-t_start)/60:.1f} min')
    print(f'  Skipped: {n_skip} | Empty ROI: {n_empty}')

    if not embs: raise RuntimeError('No embeddings extracted.')
    arr = np.array(embs)
    out = emb_dir / 'vit_swinunetr_embeddings.npz'
    np.savez_compressed(out, embeddings=arr, patient_ids=np.array(ids), timepoints=np.array(tps))
    print(f'  Saved: {out} shape={arr.shape}')

    if spatial_tokens_list:
        max_tok = max(t.shape[0] for t in spatial_tokens_list)
        C_tok = spatial_tokens_list[0].shape[1]
        padded = np.zeros((len(spatial_tokens_list), max_tok, C_tok), dtype=np.float32)
        n_tokens = np.zeros(len(spatial_tokens_list), dtype=np.int32)
        for j, tok in enumerate(spatial_tokens_list):
            padded[j, :tok.shape[0], :] = tok
            n_tokens[j] = tok.shape[0]
        tok_out = emb_dir / 'vit_spatial_tokens.npz'
        np.savez_compressed(tok_out, spatial_tokens=padded, token_counts=n_tokens,
            patient_ids=np.array(ids), timepoints=np.array(tps), bboxes=np.array(bboxes_list))
        print(f'  Spatial tokens: {tok_out} shape={padded.shape}')

    import pandas as pd
    vol_rows = []
    for j, (pid, tp, emb) in enumerate(zip(ids, tps, embs)):
        v = emb[-9:]
        vol_rows.append({
            'patient_id': pid, 'timepoint': tp,
            'wt_vol': float(np.expm1(v[0])), 'tc_vol': float(np.expm1(v[1])), 'et_vol': float(np.expm1(v[2])),
            'has_wt': float(v[3]), 'has_tc': float(v[4]), 'has_et': float(v[5]),
            'tc_wt_ratio': float(v[6]), 'et_wt_ratio': float(v[7]), 'et_tc_ratio': float(v[8]),
        })
    pd.DataFrame(vol_rows).to_csv(emb_dir / 'tumor_volumes.csv', index=False)
    print(f'  Volumes CSV: {len(vol_rows)} rows')
    return arr

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('\nExtracting SwinUNETR embeddings (ROI-crop + octant + volumetric)...')
embeddings = extract_embeddings(model)

In [ ]:
npz_path = OUTPUT_ROOT / 'embeddings' / 'vit_swinunetr_embeddings.npz'
if npz_path.exists():
    data = np.load(npz_path)
    emb_arr = data['embeddings']
    N, D = emb_arr.shape
    norms = np.linalg.norm(emb_arr, axis=1)
    emb_norm = emb_arr / (norms[:, None] + 1e-8)
    n_pairs = min(200, N*(N-1)//2)
    pairs = random.sample([(i,j) for i in range(N) for j in range(i+1,N)], n_pairs)
    sims = [float(np.dot(emb_norm[i], emb_norm[j])) for i,j in pairs]
    cos_mean = float(np.mean(sims))
    diverse = 100.0 * float(np.mean(np.array(sims) < 0.95))
    status = 'GOOD' if diverse > 20 else 'LOW'
    print(f'ViT Embedding Diversity — N={N}, D={D}')
    print(f'  Norm: [{norms.min():.2f}, {norms.max():.2f}] mean={norms.mean():.2f}')
    print(f'  Cos sim: mean={cos_mean:.3f} | {diverse:.1f}% < 0.95 [{status}]')

In [ ]:
summary = {
    'model': MODEL_NAME, 'best_dice': float(best_dice),
    'dataset': 'MU-Glioma-Post',
    'regions': REGIONS,
    'train_scans': len(train_dicts), 'val_scans': len(val_dicts),
    'all_scans': len(all_dicts),
    'pretrained_source': 'BraTS 2021 Fold 1 (original)',
    'two_stage': True,
}
(OUTPUT_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2))

print('='*55)
print(f'  SwinUNETR Fine-Tuning Complete (MU-Glioma)')
print(f'  Best Dice: {best_dice:.4f}')
print(f'  All scans (emb): {len(all_dicts)}')
print(f'  Outputs: {OUTPUT_ROOT}')
print('='*55)